# 02 Document Embeddings and FAISS

Kernel: **Python (NLP)**. This notebook uses only the frozen handoff files and saves derived outputs in the Phase 2 directories.

## Scope

Generate one normalized vector per frozen document using the exact pinned MiniLM revision, create an exact FAISS inner-product index, preserve a stable position-to-source-ID map, and save smoke-search outputs. This notebook does not read queries, qrels, answers, references, or test data.

In [1]:
from pathlib import Path
import os, json, hashlib, platform, time
import numpy as np
import pandas as pd
import faiss
import torch, sentence_transformers, transformers
from huggingface_hub import snapshot_download
from sentence_transformers import SentenceTransformer

HANDOFF_ROOT = Path.cwd().resolve()
while not (HANDOFF_ROOT / "02_FROZEN_DATA").is_dir(): HANDOFF_ROOT = HANDOFF_ROOT.parent
PHASE_ROOT = HANDOFF_ROOT / "Phase2"
BUILD_ROOT = PHASE_ROOT / "DocumentBuilding"
DATA_DIR = BUILD_ROOT / "data"; EMBEDDING_DIR = BUILD_ROOT / "embeddings"; INDEX_DIR = BUILD_ROOT / "indexes"; MANIFEST_DIR = BUILD_ROOT / "manifests"
HF_HOME = HANDOFF_ROOT / "NLP" / "hf_cache"; os.environ["HF_HOME"] = str(HF_HOME)
for folder in (EMBEDDING_DIR, INDEX_DIR, MANIFEST_DIR): folder.mkdir(parents=True, exist_ok=True)
MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"
DEVICE = "cpu"
CORPUS_PATH = HANDOFF_ROOT / "02_FROZEN_DATA" / "Corpus_4192_Gaming_Documents.csv"
DOC_MANIFEST_PATH = DATA_DIR / "chronoplay_document_manifest_v1.csv"
EMBEDDINGS_PATH = EMBEDDING_DIR / "chronoplay_minilm_document_embeddings_v1.npy"
INDEX_PATH = INDEX_DIR / "chronoplay_minilm_faiss_index_v1.faiss"
MAPPING_PATH = MANIFEST_DIR / "chronoplay_document_id_mapping_v1.csv"
CONFIG_PATH = MANIFEST_DIR / "chronoplay_minilm_document_embedding_manifest_v1.json"
for path in (EMBEDDINGS_PATH, INDEX_PATH, MAPPING_PATH, CONFIG_PATH):
    if path.exists(): raise FileExistsError(f"Refusing to overwrite {path}; select a new artifact version.")
print("Pinned model:", MODEL_ID, MODEL_REVISION, "device:", DEVICE)

/Users/vaidhav/Desktop/MyProjects/NLP/ChronoPlay_Team_Handoff_v1/NLP/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1028 15:39:17.973000 8534 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Pinned model: sentence-transformers/all-MiniLM-L6-v2 1110a243fdf4706b3f48f1d95db1a4f5529b4d41 device: cpu


In [2]:
# Reconstruct exactly the normalized document text used in notebook 01 and verify per-document hashes.
corpus = pd.read_csv(CORPUS_PATH, usecols=["id", "title", "contents"], dtype={"id":"string"})
document_manifest = pd.read_csv(DOC_MANIFEST_PATH, dtype={"id":"string"})
ingestion_report = json.loads((BUILD_ROOT / "manifests" / "chronoplay_document_ingestion_report_v1.json").read_text())
assert len(corpus) == len(document_manifest) == 4192
assert corpus["id"].tolist() == document_manifest["id"].tolist()
def normalize_newlines(value):
    import unicodedata
    return unicodedata.normalize("NFC", str(value)).replace("\r\n", "\n").replace("\r", "\n")
document_texts = [f"Title: {normalize_newlines(t)}\nContent: {normalize_newlines(c)}" for t,c in zip(corpus["title"],corpus["contents"])]
text_hashes = [hashlib.sha256(x.encode("utf-8")).hexdigest() for x in document_texts]
assert text_hashes == document_manifest["document_text_sha256"].tolist()
print("Verified source order and document text hashes for",len(document_texts),"documents")

Verified source order and document text hashes for 4192 documents


In [3]:
# Load only the approved immutable checkpoint and verify the resolved local snapshot hash.
snapshot_dir = Path(snapshot_download(repo_id=MODEL_ID,revision=MODEL_REVISION,cache_dir=str(HF_HOME),allow_patterns=["*.json","*.txt","*.safetensors"],local_files_only=True))
assert snapshot_dir.name == MODEL_REVISION
torch.set_num_threads(1)
faiss.omp_set_num_threads(1)
model = SentenceTransformer(str(snapshot_dir),device=DEVICE)
model_commit = getattr(model[0].auto_model.config,"_commit_hash",None)
assert model.max_seq_length == ingestion_report["sentence_transformer_max_seq_length"]
assert model.get_sentence_embedding_dimension() == ingestion_report["embedding_dimension"]
if model_commit is not None: assert model_commit == MODEL_REVISION
print("Model revision:",snapshot_dir.name,"config commit:",model_commit)
print("max_seq_length:",model.max_seq_length,"dimension:",model.get_sentence_embedding_dimension())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10566.29it/s]

Model revision: 1110a243fdf4706b3f48f1d95db1a4f5529b4d41 config commit: None
max_seq_length: 256 dimension: 384



/var/folders/6x/b5gh_pcd4pbb668mxxxjshjw0000gn/T/ipykernel_8534/1722408657.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  assert model.get_sentence_embedding_dimension() == ingestion_report["embedding_dimension"]
/var/folders/6x/b5gh_pcd4pbb668mxxxjshjw0000gn/T/ipykernel_8534/1722408657.py:12: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("max_seq_length:",model.max_seq_length,"dimension:",model.get_sentence_embedding_dimension())


In [4]:
# Confirm the loaded CPU model can produce finite vectors before starting the full corpus pass.
smoke_vectors = model.encode(document_texts[:2], batch_size=2, show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=False, device=DEVICE)
assert np.asarray(smoke_vectors).shape == (2, 384) and np.isfinite(smoke_vectors).all()
print("CPU encoder smoke check passed:", np.asarray(smoke_vectors).shape)

CPU encoder smoke check passed: (2, 384)


In [5]:
# Inspect the longest documents and show what the configured 256-token input window omits after the retained prefix.
long_ids = document_manifest.nlargest(3, "token_count")["id"].astype(str).tolist()
for doc_id in long_ids:
    row_index = corpus.index[corpus["id"].astype(str) == doc_id][0]
    tokens = model.tokenizer(document_texts[row_index], add_special_tokens=True, truncation=False)["input_ids"]
    print("\nDocument ID:", doc_id, "token_count:", len(tokens), "configured_limit:", model.max_seq_length)
    print("Retained text prefix (first 700 chars):\n", document_texts[row_index][:700])
    print("Text after approximate retained token boundary (decode after max sequence):\n", model.tokenizer.decode(tokens[model.max_seq_length:model.max_seq_length+80]))
print("This inspection informs truncation risk only; it does not change the one-document representation or model setting.")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2464 > 256). Running this sequence through the model will result in indexing errors



Document ID: eb838cb3-a4be-4340-acbb-cb9ff9a85ed5 token_count: 2464 configured_limit: 256
Retained text prefix (first 700 chars):
 Title: Schematics
Content: # Schematics

**Categories:** Beta, TODO

## Content

This section is intended to be updated with detailed information about various schematics available in the game. The schematics listed below represent a wide array of items, each offering unique functionalities and enhancements for players.

### List of Schematics

- **A Dart for Every Man**: A schematic for crafting a versatile dart weapon.
- **Abulurd's Rapture**: A schematic for creating a powerful item named after Abulurd.
- **Acheronian Armor Set**: Includes schematics for Acheronian Boots, Chestplate, Gauntlets, Helmet, and Pants.
- **Adaptive Holtzman Shield**: A schematic for crafting a shield with adapt
Text after approximate retained token boundary (decode after max sequence):
 reaping capabilities. - * * advanced suspensor jacket * * : a schematic for crafting a jac

In [6]:
# Encode in small CPU batches, then L2-normalize explicitly before exact inner-product search.
start_time = time.perf_counter()
raw_embeddings = model.encode(
    document_texts, batch_size=4, show_progress_bar=True, convert_to_numpy=True,
    normalize_embeddings=False, device=DEVICE,
)
raw_embeddings = np.asarray(raw_embeddings,dtype=np.float32)
assert raw_embeddings.shape == (4192,384)
norms = np.linalg.norm(raw_embeddings,axis=1,keepdims=True)
assert np.all(norms > 0)
embeddings = np.ascontiguousarray(raw_embeddings / norms,dtype=np.float32)
embedding_seconds = time.perf_counter()-start_time
normed_norms = np.linalg.norm(embeddings,axis=1)
assert np.allclose(normed_norms,1.0,atol=1e-5)
np.save(EMBEDDINGS_PATH,embeddings)
print("Embedding matrix:",embeddings.shape,embeddings.dtype,"seconds:",round(embedding_seconds,2))
print("Norm range:",float(normed_norms.min()),float(normed_norms.max()))

Batches:   0%|          | 0/1048 [00:00<?, ?it/s]

Batches:   0%|          | 1/1048 [00:00<02:09,  8.08it/s]

Batches:   0%|          | 2/1048 [00:00<02:42,  6.43it/s]

Batches:   0%|          | 4/1048 [00:00<02:08,  8.12it/s]

Batches:   0%|          | 5/1048 [00:00<02:02,  8.51it/s]

...

Batches:  99%|█████████▉| 1036/1048 [01:33<00:00, 35.15it/s]

Batches:  99%|█████████▉| 1041/1048 [01:33<00:00, 38.89it/s]

Batches: 100%|█████████▉| 1047/1048 [01:33<00:00, 44.12it/s]

Batches: 100%|██████████| 1048/1048 [01:33<00:00, 11.18it/s]

Embedding matrix: (4192, 384) float32 seconds: 93.78
Norm range: 0.9999998807907104 1.0000001192092896


In [7]:
# FAISS position order is exactly the frozen corpus row order; map each position to the original source ID.
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)
assert index.ntotal == 4192 and index.d == 384
faiss.write_index(index,str(INDEX_PATH))
mapping = pd.DataFrame({"vector_position":np.arange(len(corpus),dtype=np.int64),"document_id":corpus["id"].astype(str)})
mapping.to_csv(MAPPING_PATH,index=False)
print("FAISS index:",index.ntotal,"vectors; metric_type:",index.metric_type,"(inner product)")
print("Mapping unique IDs:",mapping["document_id"].nunique())

FAISS index: 4192 vectors; metric_type: 0 (inner product)
Mapping unique IDs: 4192


In [8]:
# Use fixed, hand-written smoke queries that are not dataset evaluation queries.
SMOKE_QUERIES = [
    {"smoke_id":"mechanics_01","text":"How does a player repair damaged equipment?"},
    {"smoke_id":"technical_01","text":"How can connection problems be troubleshot?"},
    {"smoke_id":"progression_01","text":"What are the steps to unlock a new ability?"},
]
smoke_vectors = model.encode([q["text"] for q in SMOKE_QUERIES],convert_to_numpy=True,normalize_embeddings=True,device=DEVICE)
smoke_scores,smoke_positions = index.search(np.asarray(smoke_vectors,dtype=np.float32),5)
smoke_results=[]
for query,scores,positions in zip(SMOKE_QUERIES,smoke_scores,smoke_positions):
    smoke_results.append({"smoke_id":query["smoke_id"],"query":query["text"],"top5":[{"position":int(pos),"document_id":str(mapping.iloc[int(pos)]["document_id"]),"score":float(score)} for score,pos in zip(scores,positions)]})
display(pd.DataFrame([{"smoke_id":r["smoke_id"],"top_document_ids":[x["document_id"] for x in r["top5"]]} for r in smoke_results]))

,smoke_id,top_document_ids
0,mechanics_01,"[0a88ab76-ddd2-4e0e-8314-6547bfc8ec5f, 88ab038..."
1,technical_01,"[04ddb827-8a32-4229-a931-c4dc4ca163e5, 33b6927..."
2,progression_01,"[5bb32f62-3de0-464d-8cac-c4e01458f3b8, e495147..."


In [9]:
# Save exact model, source hashes, package versions, index parameters, timings, and smoke results.
def sha256_file(path):
    digest=hashlib.sha256()
    with open(path,"rb") as stream:
        for block in iter(lambda:stream.read(1024*1024),b""):digest.update(block)
    return digest.hexdigest()
manifest={
 "status":"PASS","model_id":MODEL_ID,"model_revision":MODEL_REVISION,"loaded_snapshot_commit":snapshot_dir.name,"model_config_commit":model_commit,
 "dataset_version":"ChronoPlay-QA Claude Verified v1.1","corpus_file":"02_FROZEN_DATA/Corpus_4192_Gaming_Documents.csv",
 "corpus_sha256":sha256_file(CORPUS_PATH),"document_count":len(corpus),"document_id_order":"source CSV row order",
 "document_text":"Title: {title}\nContent: {contents}","normalization":"NFC; CRLF/CR to LF only","chunking":"none; one source document per vector",
 "model_max_seq_length":int(model.max_seq_length),"tokenizer_max_length":int(model.tokenizer.model_max_length),
 "truncation_policy":"Model's pinned configured maximum length; no manual change or chunking.",
 "embedding_dimension":int(dimension),"embedding_dtype":str(embeddings.dtype),"l2_normalized":True,
 "index_type":"faiss.IndexFlatIP","index_metric":"inner product over L2-normalized vectors (cosine similarity)","index_count":int(index.ntotal),
 "position_mapping":"chronoplay_document_id_mapping_v1.csv; position follows source row order",
 "device":DEVICE,"batch_size":4,"embedding_seconds":embedding_seconds,
 "versions":{"python":platform.python_version(),"sentence-transformers":sentence_transformers.__version__,"transformers":transformers.__version__,"torch":torch.__version__,"numpy":np.__version__,"faiss":faiss.__version__},
 "artifacts":{},"smoke_tests":smoke_results,
}
for key,path in {"embeddings":EMBEDDINGS_PATH,"index":INDEX_PATH,"mapping":MAPPING_PATH}.items():
    manifest["artifacts"][key]={"path":str(path.relative_to(HANDOFF_ROOT)),"bytes":path.stat().st_size,"sha256":sha256_file(path)}
if CONFIG_PATH.exists(): raise FileExistsError(CONFIG_PATH)
CONFIG_PATH.write_text(json.dumps(manifest,indent=2)+"\n",encoding="utf-8")
print("Saved reusable index artifacts and manifest:")
for p in (EMBEDDINGS_PATH,INDEX_PATH,MAPPING_PATH,CONFIG_PATH):print(p.relative_to(HANDOFF_ROOT),p.stat().st_size,"bytes")

Saved reusable index artifacts and manifest:
Phase2/DocumentBuilding/embeddings/chronoplay_minilm_document_embeddings_v1.npy 6439040 bytes
Phase2/DocumentBuilding/indexes/chronoplay_minilm_faiss_index_v1.faiss 6438957 bytes
Phase2/DocumentBuilding/manifests/chronoplay_document_id_mapping_v1.csv 174982 bytes
Phase2/DocumentBuilding/manifests/chronoplay_minilm_document_embedding_manifest_v1.json 4824 bytes
